In [75]:
import pandas as pd
import numpy as np
from Bio import SeqIO
import re
import subprocess
import gzip


nochim_records = SeqIO.to_dict(SeqIO.parse("/home/nanopore/projects/amplicon_workflow/results/16S_compost_SUP_SILVA_097/chimera_filtering/barcode23_nochim.fasta", "fasta"))
clust_df = pd.read_csv("/home/nanopore/projects/amplicon_workflow/reports/16S_compost_SUP_SILVA_097/clustering/barcode23_cluster_report.tsv", sep="\t", header=None)
derep_df = pd.read_csv("/home/nanopore/projects/amplicon_workflow/reports/16S_compost_SUP_SILVA_097/dereplication/barcode23_derep.tsv", sep="\t", header=None)


clust_df[[0]] = np.where(clust_df[[0]] == "C", np.nan, clust_df[[0]])
clust_df = clust_df.dropna(axis=0)

clust_df = clust_df.iloc[:, -2:]

clust_df = clust_df.rename(columns={clust_df.columns[0]: "derep_id", clust_df.columns[1]: "cluster_id"})
clust_df["cluster_id"] = np.where(clust_df["cluster_id"] == "*", clust_df["derep_id"], clust_df["cluster_id"])

def extract_before_semicolon(cell):
    if isinstance(cell, str):
        match = re.match(r'^(.*?);', cell)
        if match:
            return match.group(1)
    return cell  # Return original if no match or not a string

clust_df = clust_df.map(extract_before_semicolon)

clust_df = clust_df.groupby("cluster_id")["derep_id"].apply(set)

clust_dict = clust_df.to_dict()
print(len(clust_dict))
print(len(nochim_records))
filtered_records = {}
for key, value in clust_dict.items():
    if key in nochim_records:
        filtered_records[key] = value
# for key in nochim_records.keys():
#     if key in clust_dict:
#         filtered_records[key] = clust_dict[key]

derep_df = derep_df.iloc[:, :2]
derep_df = derep_df.rename(columns={derep_df.columns[0]: "fq_id", derep_df.columns[1]: "fa_id"})
derep_df = derep_df.groupby("fa_id")["fq_id"].apply(list)
derep_dict = derep_df.to_dict()

# Create an empty dictionary to store the results
filt_og_ids = set()

# Loop through each key and value in filtered_records
for cluster, derep_ids in filtered_records.items():
    # For each cluster, collect the corresponding original_ids from derep_dict
    original_ids = []
    for id_ in derep_ids:
        if id_ in derep_dict:
            original_ids.append(",".join(derep_dict[id_]))
    # Add the collected values to the result dictionary
    filt_og_ids.update(original_ids)


with gzip.open("/home/nanopore/projects/amplicon_workflow/results/16S_compost_SUP_SILVA_097/filtering/barcode23_filt.fastq.gz", "rt") as out_handle:
    subset_records = (record for record in SeqIO.parse(out_handle, "fastq") if record.id in filt_og_ids)
    with gzip.open("/home/nanopore/projects/amplicon_workflow/results/16S_compost_SUP_SILVA_097/filtering/barcode23_clean.fastq.gz", "wt") as out_handle:
        SeqIO.write(subset_records, out_handle, "fastq")

    
        # SeqIO.write(nochim_seqs, "/home/nanopore/projects/amplicon_workflow/results/16S_compost_SUP_SILVA_097/chimera_filtering/barcode23_nochim.fastq", "fastq")

5043
2777
